Create evaluation notebook to write the thresholds and detection result files

**Thresholds**. For each of the three statistics (T2, SPE, ridge), the threshold is numpy.quantile(scores, 0.99) over the validation runs, with NumPy’s default interpolation. Write results/thresholds.csv with columns detector, threshold.

In [3]:
# Take all the validation scores for each statistic and take the 99 percentile 
# Import Functions
import numpy as np
import polars as pl

# Read parquet files
ridge = pl.read_parquet("../results/scores_ridge.parquet")
# pca = pl.read_parquet("../results/scores_pca.parquet")

# Set up global variables
RUN = "simulationRun"
FAULT = "faultNumber"


In [4]:
# Compute the thresholds for each statistic

ridge_filtered = ridge.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400) & (pl.col(FAULT) == 0))
# get the ridge score column and convert it to a numpy array 
ridge_score = ridge_filtered.get_column("score") 
scores = ridge_score.to_numpy()
thresh_ridge = np.quantile(scores, 0.99)

'''
pca_filtered = pca.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400) & (pl.col(FAULT) == 0))
# Get the T2 and SPE score columns and convert them to numpy arrays
T2_score = pca_filtered.get_column("T2") 
T2_score = T2_score.to_numpy() 
thresh_T2 = np.quantile(T2_score, 0.99)

SPE_score = pca_filtered.get_column("SPE")
SPE_score = SPE_score.to_numpy()
thresh_SPE = np.quantile(SPE_score, 0.99)

# Create threshold result table
data = {"detector": ["T2", "SPE", "Ridge"], "threshold": [thresh_T2, thresh_SPE, thresh_ridge]}
thresholds = pl.DataFrame(data)
thresholds
'''
# Write it to results
# thresholds.write_csv("../results/thresholds.csv")

'\npca_filtered = pca.filter((pl.col(RUN) >= 301) & (pl.col(RUN) <= 400) & (pl.col(FAULT) == 0))\n# Get the T2 and SPE score columns and convert them to numpy arrays\nT2_score = pca_filtered.get_column("T2") \nT2_score = T2_score.to_numpy() \nthresh_T2 = np.quantile(T2_score, 0.99)\n\nSPE_score = pca_filtered.get_column("SPE")\nSPE_score = SPE_score.to_numpy()\nthresh_SPE = np.quantile(SPE_score, 0.99)\n\n# Create threshold result table\ndata = {"detector": ["T2", "SPE", "Ridge"], "threshold": [thresh_T2, thresh_SPE, thresh_ridge]}\nthresholds = pl.DataFrame(data)\nthresholds\n'

**Alarms**. A sample is in alarm when it and the two samples before it all exceed the threshold. Alarms never cross a run boundary.

In [27]:
def above_threshold_check(df, threshold, score): # score is "score", "T2", "SPE"
    # Go through each element in the scores table to check if score is above threshold
    new_df = df.with_columns(
        pl.when(pl.col(score) > threshold).then(True).otherwise(False).alias("above_threshold") )  
    # Sort the dataframe
    new_df = new_df.sort([FAULT, RUN, "sample"])
    # Check if it and the two samples before exceed the threshold
    check = new_df.with_columns(
        pl.col("above_threshold").shift(1).over(FAULT, RUN).alias("one_sample_above"),
        pl.col("above_threshold").shift(2).over(FAULT, RUN).alias("two_samples_above"))
    check = check.with_columns(
        pl.when((pl.col("above_threshold") == True) & (pl.col("one_sample_above") == True)
                & (pl.col("two_samples_above") == True)).then(True).otherwise(False).alias("in_alarm"))
    # Save the in_alarm results
    alarms = check.select(
        pl.col(FAULT), pl.col(RUN), pl.col("sample"), pl.col("in_alarm")
    )
    alarms = alarms.with_columns(pl.lit(score).alias("detector")) # add a detector column to identify stat
    return alarms
   

In [ ]:
# Get alarms tables 
ridge_alarms = above_threshold_check(ridge, thresh_ridge, "score")
#print(alarms)
T2_alarms = above_threshold_check(pca, thresh_T2, "T2")

SPE_alarms = above_threshold_check(pca, thresh_SPE, "SPE")

shape: (298_800, 5)
┌─────────────┬───────────────┬────────┬──────────┬──────────┐
│ faultNumber ┆ simulationRun ┆ sample ┆ in_alarm ┆ detector │
│ ---         ┆ ---           ┆ ---    ┆ ---      ┆ ---      │
│ i64         ┆ i64           ┆ i64    ┆ bool     ┆ str      │
╞═════════════╪═══════════════╪════════╪══════════╪══════════╡
│ 0           ┆ 301           ┆ 3      ┆ false    ┆ score    │
│ 0           ┆ 301           ┆ 4      ┆ false    ┆ score    │
│ 0           ┆ 301           ┆ 5      ┆ false    ┆ score    │
│ 0           ┆ 301           ┆ 6      ┆ false    ┆ score    │
│ 0           ┆ 301           ┆ 7      ┆ false    ┆ score    │
│ …           ┆ …             ┆ …      ┆ …        ┆ …        │
│ 20          ┆ 20            ┆ 496    ┆ true     ┆ score    │
│ 20          ┆ 20            ┆ 497    ┆ true     ┆ score    │
│ 20          ┆ 20            ┆ 498    ┆ true     ┆ score    │
│ 20          ┆ 20            ┆ 499    ┆ true     ┆ score    │
│ 20          ┆ 20            ┆ 500

**Metrics, for each detector:**

false-alarm rate: the share of samples in alarm over the test runs, reported as fault 0;

detection rate: for each faulty run, the share of samples after sample 20 that are in alarm, averaged over the 20 runs;

detection delay: for each faulty run, minutes from sample 20 to the first alarm after it; report the median over the runs that alarmed, and how many runs never alarmed.